<a href="https://colab.research.google.com/github/rogon666/UCB2026/blob/main/UCB_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================================
#   Maestría en Inteligencia Artificial y Gestión de Datos
#          para Finanzas, Economía y Negocios
# ----------------------------------------------------------
#            Universidad Católica Boliviana
#           Octubre/Noviembre/Diciembre 2026
# ----------------------------------------------------------
#               Aprendizaje Profundo I
# ----------------------------------------------------------
#             Rolando Gonzales Martinez
# ==========================================================
"""
Comparación de RNN, LSTM, GRU, Stacked LSTM y Stacked GRU para predecir
WTI mensual.

Instalación:
    pip install numpy pandas matplotlib scikit-learn tensorflow

Evaluación: predicción a un mes con ventana móvil de 12 meses. Cada
predicción test utiliza observaciones históricas reales disponibles hasta t-1.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping

In [1]:

# -------------------------------------------------------------
# 1. CONFIGURACIÓN
# -------------------------------------------------------------
SEMILLA = 42
VENTANA = 12       # Meses utilizados para predecir el siguiente
EPOCAS = 150
LOTE = 16

np.random.seed(SEMILLA)
tf.keras.utils.set_random_seed(SEMILLA)

# -------------------------------------------------------------
# 2. DESCARGAR Y GRAFICAR DATOS DE FRED
# -------------------------------------------------------------
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MCOILWTICO"
datos = pd.read_csv(url)
fecha = datos.columns[0]  # En FRED puede llamarse DATE u observation_date
datos[fecha] = pd.to_datetime(datos[fecha])
datos["MCOILWTICO"] = pd.to_numeric(datos["MCOILWTICO"], errors="coerce")
datos = (datos.set_index(fecha)[["MCOILWTICO"]]
         .sort_index().dropna())

# Comprobar que no se han perdido meses intermedios.
meses = datos.index.to_period("M")
if len(meses) != len(pd.period_range(meses.min(), meses.max(), freq="M")):
    raise ValueError("Hay meses faltantes en la serie: revísalos antes de crear ventanas.")

print(f"Serie: {datos.index.min().date()} a {datos.index.max().date()}")
print(f"Número de meses: {len(datos)}")
print(datos.tail())

plt.figure(figsize=(12, 4))
plt.plot(datos.index, datos["MCOILWTICO"], color="black", linewidth=1.3)
plt.title("Precio mensual del petróleo WTI (FRED: MCOILWTICO)")
plt.ylabel("USD por barril")
plt.xlabel("Fecha")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------
# 3. PARTICIÓN CRONOLÓGICA 70 % / 15 % / 15 %
# -------------------------------------------------------------
valores = datos[["MCOILWTICO"]].to_numpy(dtype=np.float32)
n = len(valores)
fin_train = int(0.70 * n)
fin_val = int(0.85 * n)

if fin_train <= VENTANA:
    raise ValueError("Muy pocas observaciones para la ventana seleccionada.")

print("\nParticiones (fechas objetivo):")
print("Entrenamiento:", datos.index[VENTANA].date(), "a", datos.index[fin_train - 1].date())
print("Validación:   ", datos.index[fin_train].date(), "a", datos.index[fin_val - 1].date())
print("Prueba:       ", datos.index[fin_val].date(), "a", datos.index[-1].date())

plt.figure(figsize=(12, 4))
plt.plot(datos.index[:fin_train], valores[:fin_train], label="Entrenamiento", color="tab:blue")
plt.plot(datos.index[fin_train:fin_val], valores[fin_train:fin_val],
         label="Validación", color="tab:orange")
plt.plot(datos.index[fin_val:], valores[fin_val:], label="Prueba", color="tab:green")
plt.title("Partición cronológica de los datos")
plt.ylabel("USD por barril")
plt.xlabel("Fecha")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# El escalador se AJUSTA únicamente sobre el entrenamiento (sin fuga de datos).
escalador = StandardScaler()
escalador.fit(valores[:fin_train])
valores_std = escalador.transform(valores)

# -------------------------------------------------------------
# 4. CREAR SECUENCIAS: 12 MESES -> 1 MES
# -------------------------------------------------------------

KeyboardInterrupt: 